<h1>HAVYN Stage 5 - Standard Model Training</h1>
<p><b>Model:</b> Qwen3-8B with 4-bit NF4 LoRA</p>
<p><b>Hardware:</b> Colab Tesla T4, FP16 compute (BF16 only on Ampere or newer)</p>
<p><b>Dataset:</b> frozen <code>havyn-train-v1</code> - 1,989 train / 223 validation / 120 heldout</p>
<p><b>Source of truth:</b> <code>havyn-stage5/scripts/train_lora.py</code> on branch <code>stage5-colab</code></p>
<p>Run the cells in order. Setup cells are rerunnable. Only the training cell starts or resumes training.</p>


In [ ]:
import os
import subprocess
import sys

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required. Use Runtime > Change runtime type > T4 GPU.")
capability = torch.cuda.get_device_capability(0)
free_bytes, total_bytes = torch.cuda.mem_get_info()
PRECISION = "bf16" if capability[0] >= 8 and torch.cuda.is_bf16_supported() else "fp16"
print("GPU:", torch.cuda.get_device_name(0))
print(f"VRAM: {total_bytes / 1024**3:.2f} GB total / {free_bytes / 1024**3:.2f} GB free")
print("Precision:", PRECISION)


In [ ]:
import hashlib
import json
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO = Path("/content/Havyn")
BRANCH = "stage5-colab"
CONFIG_PATH = REPO / "havyn-stage5/configs/train-standard.yaml"
DATASET_REPO = REPO / "havyn-stage5/datasets/havyn-train-v1"
DRIVE_ROOT = Path("/content/drive/MyDrive/havyn-stage5")
DATASET_DRIVE = DRIVE_ROOT / "datasets/havyn-train-v1"
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints/standard"
REPORT_DIR = DRIVE_ROOT / "reports/training-standard"
REPORT_PATH = REPORT_DIR / "training-report.json"
ADAPTER_DIR = CHECKPOINT_DIR / "adapter"

for path in (DATASET_DRIVE, CHECKPOINT_DIR, REPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Drive root:", DRIVE_ROOT)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Report path:", REPORT_PATH)


In [ ]:
def run(*args, cwd=None):
    result = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError("command failed: " + " ".join(str(part) for part in args))
    return result.stdout.strip()


if not (REPO / ".git").exists():
    run("git", "clone", "https://github.com/HamzaNinjaCoder1/Havyn.git", str(REPO))
else:
    run("git", "fetch", "origin", cwd=str(REPO))

dirty = run("git", "status", "--porcelain", cwd=str(REPO))
if dirty:
    print("Discarding local Colab edits to match origin:")
    print(dirty)

run("git", "checkout", "-f", "-B", BRANCH, "origin/" + BRANCH, cwd=str(REPO))
head = run("git", "rev-parse", "HEAD", cwd=str(REPO))
remote = run("git", "rev-parse", "origin/" + BRANCH, cwd=str(REPO))
if head != remote:
    raise RuntimeError(f"HEAD {head} does not match origin/{BRANCH} {remote}")
print("Branch:", BRANCH)
print("Commit:", head)


In [ ]:
REQUIREMENTS = REPO / "havyn-stage5/requirements-colab.txt"
INSTALL_MARKER = Path("/content/.havyn_requirements_sha256")
requirements_sha = hashlib.sha256(REQUIREMENTS.read_bytes()).hexdigest()
installed_sha = INSTALL_MARKER.read_text().strip() if INSTALL_MARKER.exists() else ""

if installed_sha != requirements_sha:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(REQUIREMENTS)],
        cwd=str(REPO), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout[-4000:])
    if result.returncode != 0:
        raise RuntimeError(f"Dependency installation failed with exit code {result.returncode}.")
    INSTALL_MARKER.write_text(requirements_sha)
else:
    print("Dependencies already installed for these requirements.")

import accelerate
import bitsandbytes
import datasets
import peft
import transformers
import trl

for name, module in (("transformers", transformers), ("trl", trl), ("peft", peft), ("datasets", datasets), ("accelerate", accelerate), ("bitsandbytes", bitsandbytes)):
    print(f"{name}: {module.__version__}")


In [ ]:
FROZEN_FILES = ("train.jsonl", "validation.jsonl", "heldout.jsonl", "audit-report.json", "dataset-card.json")
EXPECTED_ROWS = {"train.jsonl": 1989, "validation.jsonl": 223, "heldout.jsonl": 120}


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_dataset(root):
    manifest = json.loads((root / "FROZEN.json").read_text(encoding="utf-8"))
    rows = []
    failures = []
    for name in FROZEN_FILES:
        expected = manifest["sha256"].get(name, "MISSING")
        actual = sha256_file(root / name) if (root / name).exists() else "MISSING"
        ok = actual == expected
        rows.append((name, expected, actual, "PASS" if ok else "FAIL"))
        if not ok:
            failures.append(name)
    for name, count_expected in EXPECTED_ROWS.items():
        with (root / name).open(encoding="utf-8") as handle:
            count_actual = sum(1 for line in handle if line.strip())
        ok = count_actual == count_expected
        rows.append((name + " rows", str(count_expected), str(count_actual), "PASS" if ok else "FAIL"))
        if not ok:
            failures.append(name + " rows")
    return manifest, rows, failures


In [ ]:
DATASET_MANIFEST, dataset_rows, dataset_failures = verify_dataset(DATASET_REPO)
print(f"{'Artifact':28} {'Expected':64} {'Actual':64} Result")
for name, expected, actual, result in dataset_rows:
    print(f"{name:28} {str(expected):64} {str(actual):64} {result}")
if dataset_failures:
    raise RuntimeError("DATASET INTEGRITY: FAILED -> " + ", ".join(dataset_failures))
print("DATASET INTEGRITY: PASS")
print("frozen_at:", DATASET_MANIFEST["frozen_at"])


In [ ]:
for name in (*FROZEN_FILES, "FROZEN.json"):
    source = DATASET_REPO / name
    target = DATASET_DRIVE / name
    if not target.exists() or sha256_file(target) != sha256_file(source):
        shutil.copy2(source, target)
        print("synced:", name)

drive_manifest, drive_rows, drive_failures = verify_dataset(DATASET_DRIVE)
if drive_failures:
    raise RuntimeError("DRIVE DATASET: FAILED -> " + ", ".join(drive_failures))
if drive_manifest["sha256"] != DATASET_MANIFEST["sha256"]:
    raise RuntimeError("DRIVE DATASET: manifest differs from the repository copy.")
DATASET_OK = True
print("DRIVE DATASET: PASS")


In [ ]:
import yaml

sys.path.insert(0, str(REPO / "havyn-stage5/scripts"))
from train_lora import latest_valid_checkpoint

with CONFIG_PATH.open(encoding="utf-8") as handle:
    CONFIG = yaml.safe_load(handle)
TRAIN = CONFIG["train"]
BATCH = int(TRAIN["per_device_train_batch_size"])
ACCUM = int(TRAIN["gradient_accumulation_steps"])
EPOCHS = int(TRAIN["epochs"])
RESUME_TARGET = latest_valid_checkpoint(CHECKPOINT_DIR)
ADAPTER_FILES = sorted(path.name for path in ADAPTER_DIR.glob("adapter_model.*"))
REPORT = json.loads(REPORT_PATH.read_text(encoding="utf-8")) if REPORT_PATH.exists() else {}
COMPLETED = REPORT.get("status") == "completed" and bool(ADAPTER_FILES)
CONFIG_SHA = sha256_file(CONFIG_PATH)

print("Base model:", CONFIG["base_model"])
print("Sequence length:", CONFIG["max_seq_length"])
print("Batch size:", BATCH)
print("Gradient accumulation:", ACCUM)
print("Effective batch:", BATCH * ACCUM)
print("Epochs:", EPOCHS)
print("Learning rate:", TRAIN["learning_rate"])
print("Precision:", PRECISION)
print("GPU:", torch.cuda.get_device_name(0))
print(f"VRAM: {total_bytes / 1024**3:.2f} GB total / {free_bytes / 1024**3:.2f} GB free")
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Valid checkpoint:", RESUME_TARGET)
print("Completed adapter:", COMPLETED)
CONFIG_OK = True


In [ ]:
def run_stream(command, cwd):
    print("Command:", " ".join(str(part) for part in command), flush=True)
    environment = dict(os.environ, PYTHONUNBUFFERED="1")
    process = subprocess.Popen(
        [str(part) for part in command],
        cwd=str(cwd), text=True, bufsize=1,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=environment,
    )
    transcript = []
    for line in process.stdout:
        print(line, end="", flush=True)
        transcript.append(line)
    process.wait()
    return process.returncode, "".join(transcript)


PREFLIGHT_COMMAND = [
    sys.executable, "havyn-stage5/scripts/train_lora.py",
    "--config", str(CONFIG_PATH),
    "--dataset-dir", str(DATASET_DRIVE),
    "--output-dir", str(CHECKPOINT_DIR),
    "--preflight",
]
preflight_code, preflight_output = run_stream(PREFLIGHT_COMMAND, REPO)
if preflight_code != 0:
    print("PRE-FLIGHT: FAILED")
    print(preflight_output[-8000:])
    raise RuntimeError(f"Preflight failed with exit code {preflight_code}.")
PREFLIGHT_OK = True
print("PRE-FLIGHT: PASS")


In [ ]:
import time

drive_manifest, drive_rows, drive_failures = verify_dataset(DATASET_DRIVE)
for attempt in range(5):
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    if free_bytes >= 9 * 1024**3:
        break
    time.sleep(2)
process_list = subprocess.run(["ps", "-eo", "pid,args"], text=True, capture_output=True)
conflicts = [line.strip() for line in process_list.stdout.splitlines() if "train_lora.py" in line]
gates = {
    "dataset integrity": DATASET_OK and not drive_failures,
    "preflight": PREFLIGHT_OK,
    "configuration": CONFIG_OK and sha256_file(CONFIG_PATH) == CONFIG_SHA,
    "cuda": torch.cuda.is_available(),
    "free vram >= 9 GB": free_bytes >= 9 * 1024**3,
    "no training process": not conflicts,
}
for name, passed in gates.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")
for line in conflicts:
    print(line)
if not all(gates.values()):
    raise RuntimeError("ALL TRAINING GATES: FAILED")
print("Checkpoint state:", "completed" if COMPLETED else (f"resume from {RESUME_TARGET}" if RESUME_TARGET else "fresh start"))
print("ALL TRAINING GATES: PASS")


In [ ]:
if not (DATASET_OK and PREFLIGHT_OK and CONFIG_OK):
    raise RuntimeError("Training blocked: upstream gates did not pass.")

if COMPLETED:
    TRAINING_STATE = "COMPLETED"
    print("A completed adapter already exists. Training was not restarted.")
else:
    TRAINING_COMMAND = [
        sys.executable, "havyn-stage5/scripts/train_lora.py",
        "--config", str(CONFIG_PATH),
        "--dataset-dir", str(DATASET_DRIVE),
        "--output-dir", str(CHECKPOINT_DIR),
    ]
    if RESUME_TARGET is not None:
        TRAINING_COMMAND.append("--resume")
        print("Resuming from:", RESUME_TARGET)
    else:
        print("Starting from a fresh checkpoint directory.")
    training_code, training_output = run_stream(TRAINING_COMMAND, REPO)
    REPORT = json.loads(REPORT_PATH.read_text(encoding="utf-8")) if REPORT_PATH.exists() else {}
    ADAPTER_FILES = sorted(path.name for path in ADAPTER_DIR.glob("adapter_model.*"))
    if training_code == 0 and REPORT.get("status") == "completed" and ADAPTER_FILES:
        TRAINING_STATE = "COMPLETED"
        print("TRAINING: COMPLETED")
    else:
        TRAINING_STATE = "FAILED"
        print("TRAINING: FAILED")
        print("Exit code:", training_code)
        print("Report status:", REPORT.get("status"))
        print("Adapter files:", ADAPTER_FILES)
        print(training_output[-8000:])


In [ ]:
REPORT = json.loads(REPORT_PATH.read_text(encoding="utf-8")) if REPORT_PATH.exists() else {}
CHECKPOINTS = sorted(
    (path for path in CHECKPOINT_DIR.glob("checkpoint-*") if path.is_dir()),
    key=lambda path: path.stat().st_mtime,
)
ADAPTER_FILES = sorted(path.name for path in ADAPTER_DIR.glob("adapter_model.*"))
print("Report status:", REPORT.get("status"))
print("Started:", REPORT.get("started"))
print("Finished:", REPORT.get("finished"))
print("Final loss:", REPORT.get("final_loss"))
print("Resumed from:", REPORT.get("resumed_from"))
print("Checkpoints:", [path.name for path in CHECKPOINTS])
print("Adapter directory:", ADAPTER_DIR)
print("Adapter files:", ADAPTER_FILES)


In [ ]:
print("HAVYN STAGE 5 TRAINING RESULT")
print("Dataset:", "PASS" if DATASET_OK else "FAILED")
print("Preflight:", "PASS" if PREFLIGHT_OK else "FAILED")
print("Training:", TRAINING_STATE)
print("Checkpoint:", str(CHECKPOINTS[-1]) if CHECKPOINTS else "none")
print("Adapter:", str(ADAPTER_DIR) if ADAPTER_FILES else "none")
print("Final loss:", REPORT.get("final_loss"))
